In [1]:
# In[1]:


# # Composite DNA Decoder: Training & Evaluation - Cross-Platform Robustness
# ## Nanopore (R21, B22, NP22, NPF22) + Newer Illumina (BOS22)
# ## Each profile uses its standard sequence length from the corresponding original dataset

# In[1]:

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# In[2]:

# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080


In [2]:
# In[3]:

# =============================================================================
# CELL 3: CONFIGURATION & HYPERPARAMETERS
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
# NEW PLATFORM OPTIONS (cross-platform robustness study):
#   "R21"    -> Oxford Nanopore MinION + Twist (Rang et al. 2021)
#   "B22"    -> Nanopore MinION Short + Twist  (Bar-Lev et al. 2022)
#   "BOS22"  -> Illumina MiSeq 2022 + Twist   (very low error, newer Illumina)
#   "NP22"   -> Nanopore Pilot Nov-2022 + Twist (highly non-uniform across bases)
#   "NPF22"  -> Nanopore Full Pool Nov-2022 + Twist (comprehensive Nanopore)
# ----------------------------------------------------------
ERROR_MODEL = "NP22"  # <-- CHANGE THIS

# ------------------- SELECT ALPHABET MODE -------------------
# Options: "2mix_only", "2mix_3mix", "2mix_3mix_4mix"
ALPHABET_MODE = "2mix_3mix_4mix"  # <-- CHANGE THIS
# ------------------------------------------------------------

# Dataset parameters
NUM_SAMPLES = 100000
MAX_COVERAGE = 25

# Error model specifications - standard sequence lengths from original datasets
# All new profiles use the same Erlich/Twist 152nt oligo pool (16nt index → n=136)
# Combined with original profiles (EZ17 n=136, G15 n=104, O17 n=77), this gives
# standard sequence lengths n ∈ {77, 104, 136} across the full set of 8 profiles.
ERROR_MODEL_SPECS = {
    "R21": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "R21",
        "platform": "Oxford Nanopore MinION",
        "synthesis": "Twist Bioscience"
    },
    "B22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "B22",
        "platform": "Nanopore MinION Short",
        "synthesis": "Twist Bioscience"
    },
    "BOS22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "BOS22",
        "platform": "Illumina MiSeq 2022",
        "synthesis": "Twist Bioscience"
    },
    "NP22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NP22",
        "platform": "Nanopore Pilot Nov-2022",
        "synthesis": "Twist Bioscience"
    },
    "NPF22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NPF22",
        "platform": "Nanopore Full Pool Nov-2022",
        "synthesis": "Twist Bioscience"
    },
}

# Vocabulary sizes
VOCAB_SIZES = {
    "2mix_only": 10,
    "2mix_3mix": 14,
    "2mix_3mix_4mix": 15
}

# Build configuration
CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_name": ERROR_MODEL_SPECS[ERROR_MODEL]["name"],
    "platform": ERROR_MODEL_SPECS[ERROR_MODEL]["platform"],
    
    # Alphabet Mode
    "alphabet_mode": ALPHABET_MODE,
    
    # Data Paths (matches dataset_generator_cross_platform.py output)
    "dataset_dir": "./dataset_cross_platform",
    "dataset_name": f"dna_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_{ALPHABET_MODE}",
    
    # Results directory
    "results_dir": f"./results/{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_{ALPHABET_MODE}",
    
    # Vocabulary
    "vocab_size": VOCAB_SIZES[ALPHABET_MODE],
    
    # Sequence Parameters
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    
    # Experiment Parameters
    "coverage_levels": [1, 2, 3, 5, 8, 10, 15, 20, 25],
    
    # Model Architecture (same as original for fair comparison)
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Training Parameters
    "batch_size": 500,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "patience": 10,
    "warmup_epochs": 10,
    "min_lr": 1e-6,
    
    # Reproducibility
    "seed": 42
}

# Complete dataset path
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{CONFIG['dataset_name']}_"
                          f"{NUM_SAMPLES}_{MAX_COVERAGE}.pkl")

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(f"{'='*70}")
print(f"📋 CROSS-PLATFORM CONFIGURATION")
print(f"{'='*70}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Platform: {CONFIG['platform']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Alphabet Mode: {CONFIG['alphabet_mode']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Results Dir: {CONFIG['results_dir']}")
print(f"{'='*70}")

📋 CROSS-PLATFORM CONFIGURATION
   Error Model: NP22 (NP22)
   Platform: Nanopore Pilot Nov-2022
   Sequence Length: 136
   Alphabet Mode: 2mix_3mix_4mix
   Vocab Size: 15 classes
   Dataset Path: ./dataset_cross_platform/dna_NP22_2mix_3mix_4mix_100000_25.pkl
   Results Dir: ./results/NP22_2mix_3mix_4mix


In [3]:
# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")

🎲 Random seed set to: 42


In [4]:
# In[5]:

# =============================================================================
# CELL 5: SYMBOL MAPPINGS & IDEAL VECTORS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 5
# Copy: build_symbol_to_idx(), build_ideal_vectors()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def build_symbol_to_idx(mode):
    """Build symbol-to-index mapping based on alphabet mode."""
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    symbol_to_idx.update({'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9})
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        symbol_to_idx.update({'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13})
    if mode == "2mix_3mix_4mix":
        symbol_to_idx.update({'Q1': 14})
    return symbol_to_idx


def build_ideal_vectors(mode):
    """Build ideal frequency vectors for all symbols."""
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        [0.5, 0.0, 0.0, 0.5],  # M1
        [0.0, 0.5, 0.5, 0.0],  # M2
        [0.0, 0.5, 0.0, 0.5],  # M3
        [0.0, 0.0, 0.5, 0.5],  # M4
        [0.5, 0.5, 0.0, 0.0],  # M5
        [0.5, 0.0, 0.5, 0.0],  # M6
    ]
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        third = 1.0 / 3.0
        ideal_vectors.extend([
            [third, third, third, 0.0],
            [third, third, 0.0, third],
            [third, 0.0, third, third],
            [0.0, third, third, third],
        ])
    if mode == "2mix_3mix_4mix":
        ideal_vectors.append([0.25, 0.25, 0.25, 0.25])
    return torch.tensor(ideal_vectors, dtype=torch.float32)


# Build mappings
SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"]).to(device)

print(f"\n📊 Symbol Mappings ({CONFIG['alphabet_mode']}):")
print(f"   {'Symbol':<8} {'Index':<6} {'Ideal Vector [A, C, G, T]'}")
print(f"   {'-'*50}")
for sym, idx in sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1]):
    vec = IDEAL_VECTORS[idx].cpu().numpy()
    print(f"   {sym:<8} {idx:<6} [{vec[0]:.4f}, {vec[1]:.4f}, {vec[2]:.4f}, {vec[3]:.4f}]")



📊 Symbol Mappings (2mix_3mix_4mix):
   Symbol   Index  Ideal Vector [A, C, G, T]
   --------------------------------------------------
   A        0      [1.0000, 0.0000, 0.0000, 0.0000]
   C        1      [0.0000, 1.0000, 0.0000, 0.0000]
   G        2      [0.0000, 0.0000, 1.0000, 0.0000]
   T        3      [0.0000, 0.0000, 0.0000, 1.0000]
   M1       4      [0.5000, 0.0000, 0.0000, 0.5000]
   M2       5      [0.0000, 0.5000, 0.5000, 0.0000]
   M3       6      [0.0000, 0.5000, 0.0000, 0.5000]
   M4       7      [0.0000, 0.0000, 0.5000, 0.5000]
   M5       8      [0.5000, 0.5000, 0.0000, 0.0000]
   M6       9      [0.5000, 0.0000, 0.5000, 0.0000]
   T1       10     [0.3333, 0.3333, 0.3333, 0.0000]
   T2       11     [0.3333, 0.3333, 0.0000, 0.3333]
   T3       12     [0.3333, 0.0000, 0.3333, 0.3333]
   T4       13     [0.0000, 0.3333, 0.3333, 0.3333]
   Q1       14     [0.2500, 0.2500, 0.2500, 0.2500]


In [5]:
# In[6]:

# =============================================================================
# CELL 6: DATA PREPROCESSING
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 6
# Copy: preprocess_cluster_to_matrix()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix."""
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
    return profile_matrix


In [6]:
# In[7]:

# =============================================================================
# CELL 7: PYTORCH DATASET CLASS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 7
# Copy: CompositeDNADataset class
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

class CompositeDNADataset(Dataset):
    """PyTorch Dataset for Composite DNA data."""
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


In [7]:
# In[8]:

# =============================================================================
# CELL 8: NEURAL NETWORK MODEL (Bi-LSTM)
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 8
# Copy: CompositeDecoderLSTM class
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM Decoder for Composite DNA."""
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, 4, L) -> (B, L, 4)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)  # (B, vocab_size, L)



In [8]:
# In[9]:

# =============================================================================
# CELL 9: BASELINE DECODERS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 9
# Copy: min_distance_decoder(), kl_divergence_decoder(), maximum_likelihood_decoder()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder (L2 norm)."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder with epsilon smoothing."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder with epsilon smoothing."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)


In [9]:
# In[10]:

# =============================================================================
# CELL 10: EARLY STOPPING CLASS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 10
# Copy: EarlyStopping class
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

class EarlyStopping:
    """Early stopping with patience and best model saving."""
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.path = path
        self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
            
    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"      ✓ Val loss improved ({self.best_val_loss:.4f} → {val_loss:.4f}). Saving...")
        torch.save(model.state_dict(), self.path)
        self.best_val_loss = val_loss


In [10]:
# In[11]:

# =============================================================================
# CELL 11: TRAINING FUNCTION
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 11
# Copy: train_model() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def train_model(model, train_loader, val_loader, config, weights_path, device):
    """Train the model with warmup + cosine annealing scheduler."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=config['warmup_epochs'])
    cosine_scheduler = CosineAnnealingLR(
        optimizer, T_max=config['epochs'] - config['warmup_epochs'], eta_min=config['min_lr']
    )
    scheduler = SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[config['warmup_epochs']]
    )
    
    early_stopper = EarlyStopping(patience=config['patience'], path=weights_path, verbose=True)
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    print(f"\n   🏋️ Training Configuration:")
    print(f"      Epochs: {config['epochs']}, Patience: {config['patience']}")
    print(f"      Warmup: {config['warmup_epochs']} epochs")
    print(f"      LR: {config['learning_rate']} → {config['min_lr']}")
    
    for epoch in range(config['epochs']):
        start_time = time.time()
        
        # --- TRAINING ---
        model.train()
        train_loss_accum = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_accum += loss.item()
        avg_train_loss = train_loss_accum / len(train_loader)
        
        # --- VALIDATION ---
        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss_accum += criterion(outputs, labels).item()
        avg_val_loss = val_loss_accum / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start_time
        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.2e} | Time: {elapsed:.1f}s")
        
        scheduler.step()
        early_stopper(avg_val_loss, model)
        if early_stopper.early_stop:
            print(f"\n   🛑 Early stopping triggered at epoch {epoch+1}")
            break
    
    return history


In [11]:
# In[12]:

# =============================================================================
# CELL 12: EVALUATION FUNCTION
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: Train_evaluate_2mix_3mix_4mix-Erlich.py → Cell 12
# Copy: evaluate_all_decoders() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def evaluate_all_decoders(model, loader, ideal_vectors, device):
    """Evaluate all 4 decoders on the given data loader."""
    model.eval()
    correct = {'lstm': 0, 'mindist': 0, 'kl': 0, 'ml': 0}
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)  # (B, L, 4)
            
            pred_lstm = torch.argmax(model(inputs), dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            total += labels.numel()
            correct['lstm'] += (pred_lstm == labels).sum().item()
            correct['mindist'] += (pred_mindist == labels).sum().item()
            correct['kl'] += (pred_kl == labels).sum().item()
            correct['ml'] += (pred_ml == labels).sum().item()
    
    return {k: 100 * v / total for k, v in correct.items()}



In [12]:
# In[13]:

# =============================================================================
# CELL 13: FULL EXPERIMENT FOR SINGLE COVERAGE
# =============================================================================
# Minor modifications: updated path naming and print statements for platform info

def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors, device):
    """Run complete experiment for a single coverage level."""
    print(f"\n{'='*70}")
    print(f"🔬 EXPERIMENT FOR COVERAGE M = {coverage_M}")
    print(f"   Error Model: {config['error_name']} ({config['platform']})")
    print(f"   Seq Length: {config['seq_length']}")
    print(f"{'='*70}")
    
    # 1. Prepare Data
    set_seed(config['seed'])
    
    full_ds = CompositeDNADataset(
        config['dataset_path'], config['seq_length'],
        symbol_to_idx, limit_coverage=coverage_M
    )
    
    train_size = int(0.8 * len(full_ds))
    val_size = len(full_ds) - train_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    
    print(f"   📊 Data: {train_size:,} train | {val_size:,} validation")
    print(f"   📏 Sequence length: {config['seq_length']}")
    
    # 2. Setup Model
    model = CompositeDecoderLSTM(config).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🧠 Model: {config['vocab_size']} classes, {num_params:,} parameters")
    
    # 3. Paths
    model_prefix = f"{config['error_name']}_{config['alphabet_mode']}"
    best_weights_path = os.path.join(config['results_dir'], f"best_model_{model_prefix}_M{coverage_M}.pth")
    final_weights_path = os.path.join(config['results_dir'], f"final_model_{model_prefix}_M{coverage_M}.pth")
    history_path = os.path.join(config['results_dir'], f"training_history_{model_prefix}_M{coverage_M}.json")
    
    # 4. Train
    history = train_model(model, train_loader, val_loader, config, best_weights_path, device)
    
    torch.save(model.state_dict(), final_weights_path)
    print(f"   💾 Final model saved: {final_weights_path}")
    
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=4)
    print(f"   📊 Training history saved: {history_path}")
    
    # 5. Load Best Model & Evaluate
    print(f"\n   📈 Evaluating all decoders...")
    model.load_state_dict(torch.load(best_weights_path, map_location=device))
    
    accuracies = evaluate_all_decoders(model, val_loader, ideal_vectors, device)
    
    print(f"\n   ✅ RESULTS M={coverage_M} ({config['error_name']}, {config['platform']}):")
    print(f"      Bi-LSTM:         {accuracies['lstm']:.2f}%")
    print(f"      Min. Distance:   {accuracies['mindist']:.2f}%")
    print(f"      KL Divergence:   {accuracies['kl']:.2f}%")
    print(f"      Max. Likelihood: {accuracies['ml']:.2f}%")
    
    return accuracies, history


In [13]:
# In[14]:

# =============================================================================
# CELL 14: PLOTTING FUNCTIONS
# =============================================================================

def plot_training_history(history, coverage_M, save_path, config):
    """Plot training and validation loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Loss (M={coverage_M}, {config["error_name"]} - {config["platform"]})', fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['lr'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Learning Rate', fontsize=12)
    ax2.set_title(f'LR Schedule (M={coverage_M})', fontsize=13)
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📈 Training plot saved: {save_path}")


def plot_comparison_results(results, save_path, config):
    """Plot comparison of all decoders across coverage levels."""
    plt.figure(figsize=(12, 7))
    
    plt.plot(results['coverage'], results['lstm'], 
             'o-', lw=2.5, ms=8, c='#2ecc71', label='Bi-LSTM (Ours)')
    plt.plot(results['coverage'], results['mindist'], 
             's--', lw=2.5, ms=8, c='#e74c3c', label='Min. Distance')
    plt.plot(results['coverage'], results['kl'], 
             '^-.', lw=2.5, ms=8, c='#3498db', label='KL Divergence')
    plt.plot(results['coverage'], results['ml'], 
             'd:', lw=2.5, ms=8, c='#9b59b6', label='Max. Likelihood')
    
    title = (f"Composite DNA Decoding: {config['error_name']} ({config['platform']})\n"
             f"({config['alphabet_mode']}: {config['vocab_size']} classes, "
             f"Seq Length: {config['seq_length']})")
    
    plt.xlabel("Coverage Depth (M)", fontsize=12)
    plt.ylabel("Symbol Accuracy (%)", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)
    plt.xticks(results['coverage'])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Comparison plot saved: {save_path}")


In [14]:
# In[15]:

# =============================================================================
# CELL 15: VERIFY DATASET EXISTS
# =============================================================================

print("\n" + "="*70)
print("📦 LOADING DATASET")
print("="*70)

if not os.path.exists(CONFIG['dataset_path']):
    raise FileNotFoundError(
        f"\n❌ Dataset not found: {CONFIG['dataset_path']}\n"
        f"   Please run dataset_generator_cross_platform.py with:\n"
        f"   ERROR_MODEL = '{CONFIG['error_model']}'\n"
        f"   ALPHABET_MODE = '{CONFIG['alphabet_mode']}'"
    )

with open(CONFIG['dataset_path'], 'rb') as f:
    data = pickle.load(f)

print(f"✅ Dataset loaded: {CONFIG['dataset_path']}")
print(f"   Samples: {len(data['data']):,}")
print(f"   Sequence Length: {data['metadata']['seq_length']}")
print(f"   Error Model: {CONFIG['error_name']} ({CONFIG['platform']})")
print(f"\n   Metadata:")
for key, value in data['metadata'].items():
    if key not in ['symbol_to_idx', 'symbols', 'ideal_vectors', 'error_summary']:
        print(f"      {key}: {value}")



📦 LOADING DATASET
✅ Dataset loaded: ./dataset_cross_platform/dna_NP22_2mix_3mix_4mix_100000_25.pkl
   Samples: 100,000
   Sequence Length: 136
   Error Model: NP22 (Nanopore Pilot Nov-2022)

   Metadata:
      type: Composite DNA (2mix_3mix_4mix)
      error_profile: NP22 (Nanopore Pilot Nov-2022)
      error_model: NP22
      platform: Nanopore Pilot Nov-2022
      synthesis: Twist Bioscience
      reference: Bar-Lev & Sabary et al. 2022
      full_length: 152
      index_length: 16
      num_samples: 100000
      seq_length: 136
      coverage_depth: 25
      vocab_size: 15
      alphabet_mode: 2mix_3mix_4mix
      timestamp: 2026-02-14 16:38:55
      seed: 42


In [15]:
# In[16]:

# =============================================================================
# CELL 16: BUILD SYMBOL MAPPINGS
# =============================================================================

SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"]).to(device)

print(f"\n📊 Symbol Mappings ({CONFIG['alphabet_mode']}):")
print(f"   Total symbols: {len(SYMBOL_TO_IDX)}")


📊 Symbol Mappings (2mix_3mix_4mix):
   Total symbols: 15


In [16]:
# In[17]:

# =============================================================================
# CELL 17: MAIN EXECUTION - RUN ALL EXPERIMENTS
# =============================================================================

print("\n" + "="*70)
print("🚀 RUNNING CROSS-PLATFORM EXPERIMENTS FOR ALL COVERAGE LEVELS")
print("="*70)
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Platform: {CONFIG['platform']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Coverage Levels: {CONFIG['coverage_levels']}")
print(f"   Alphabet: {CONFIG['alphabet_mode']} ({CONFIG['vocab_size']} classes)")

results = {
    'coverage': CONFIG['coverage_levels'],
    'lstm': [],
    'mindist': [],
    'kl': [],
    'ml': [],
    'config': {
        'error_model': CONFIG['error_model'],
        'error_name': CONFIG['error_name'],
        'platform': CONFIG['platform'],
        'seq_length': CONFIG['seq_length'],
        'alphabet_mode': CONFIG['alphabet_mode'],
        'vocab_size': CONFIG['vocab_size'],
        'hidden_dim': CONFIG['hidden_dim'],
        'num_layers': CONFIG['num_layers'],
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
}

all_histories = {}

for M in CONFIG['coverage_levels']:
    accuracies, history = run_experiment_for_coverage(
        M, CONFIG, SYMBOL_TO_IDX, IDEAL_VECTORS, device
    )
    
    results['lstm'].append(accuracies['lstm'])
    results['mindist'].append(accuracies['mindist'])
    results['kl'].append(accuracies['kl'])
    results['ml'].append(accuracies['ml'])
    all_histories[M] = history
    
    # Plot training history
    plot_prefix = f"{CONFIG['error_name']}_{CONFIG['alphabet_mode']}"
    plot_path = os.path.join(CONFIG['results_dir'], f"training_plot_{plot_prefix}_M{M}.png")
    plot_training_history(history, M, plot_path, CONFIG)



🚀 RUNNING CROSS-PLATFORM EXPERIMENTS FOR ALL COVERAGE LEVELS
   Error Model: NP22 (NP22)
   Platform: Nanopore Pilot Nov-2022
   Sequence Length: 136
   Coverage Levels: [1, 2, 3, 5, 8, 10, 15, 20, 25]
   Alphabet: 2mix_3mix_4mix (15 classes)

🔬 EXPERIMENT FOR COVERAGE M = 1
   Error Model: NP22 (Nanopore Pilot Nov-2022)
   Seq Length: 136
   📊 Data: 80,000 train | 20,000 validation
   📏 Sequence length: 136
   🧠 Model: 15 classes, 536,335 parameters

   🏋️ Training Configuration:
      Epochs: 100, Patience: 10
      Warmup: 10 epochs
      LR: 0.001 → 1e-06
   Epoch 001/100 | Train: 2.6695 | Val: 2.5784 | LR: 1.00e-04 | Time: 40.2s
      ✓ Val loss improved (inf → 2.5784). Saving...
   Epoch 002/100 | Train: 2.4928 | Val: 2.4606 | LR: 1.90e-04 | Time: 39.4s
      ✓ Val loss improved (2.5784 → 2.4606). Saving...
   Epoch 003/100 | Train: 2.4541 | Val: 2.4400 | LR: 2.80e-04 | Time: 40.1s
      ✓ Val loss improved (2.4606 → 2.4400). Saving...
   Epoch 004/100 | Train: 2.4316 | Val: 2.4

/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:149: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 2.4090 | Val: 2.4093 | LR: 1.00e-03 | Time: 40.8s
      ✓ Val loss improved (2.4098 → 2.4093). Saving...
   Epoch 012/100 | Train: 2.4086 | Val: 2.4096 | LR: 1.00e-03 | Time: 40.5s
      EarlyStopping counter: 1/10
   Epoch 013/100 | Train: 2.4082 | Val: 2.4094 | LR: 9.99e-04 | Time: 40.3s
      EarlyStopping counter: 2/10
   Epoch 014/100 | Train: 2.4080 | Val: 2.4090 | LR: 9.97e-04 | Time: 40.8s
      ✓ Val loss improved (2.4093 → 2.4090). Saving...
   Epoch 015/100 | Train: 2.4079 | Val: 2.4093 | LR: 9.95e-04 | Time: 40.2s
      EarlyStopping counter: 1/10
   Epoch 016/100 | Train: 2.4076 | Val: 2.4089 | LR: 9.92e-04 | Time: 40.2s
      ✓ Val loss improved (2.4090 → 2.4089). Saving...
   Epoch 017/100 | Train: 2.4074 | Val: 2.4088 | LR: 9.89e-04 | Time: 40.3s
      ✓ Val loss improved (2.4089 → 2.4088). Saving...
   Epoch 018/100 | Train: 2.4072 | Val: 2.4089 | LR: 9.85e-04 | Time: 40.5s
      EarlyStopping counter: 1/10
   Epoch 019/100 | Train: 2.4071 | V

   Epoch 021/100 | Train: 2.1618 | Val: 2.1623 | LR: 9.70e-04 | Time: 70.0s
      ✓ Val loss improved (2.1628 → 2.1623). Saving...
   Epoch 022/100 | Train: 2.1613 | Val: 2.1603 | LR: 9.64e-04 | Time: 70.1s
      ✓ Val loss improved (2.1623 → 2.1603). Saving...
   Epoch 023/100 | Train: 2.1597 | Val: 2.1607 | LR: 9.57e-04 | Time: 70.0s
      EarlyStopping counter: 1/10
   Epoch 024/100 | Train: 2.1592 | Val: 2.1598 | LR: 9.49e-04 | Time: 70.1s
      ✓ Val loss improved (2.1603 → 2.1598). Saving...
   Epoch 025/100 | Train: 2.1584 | Val: 2.1586 | LR: 9.42e-04 | Time: 69.8s
      ✓ Val loss improved (2.1598 → 2.1586). Saving...
   Epoch 026/100 | Train: 2.1578 | Val: 2.1580 | LR: 9.33e-04 | Time: 69.4s
      ✓ Val loss improved (2.1586 → 2.1580). Saving...
   Epoch 027/100 | Train: 2.1572 | Val: 2.1596 | LR: 9.24e-04 | Time: 69.9s
      EarlyStopping counter: 1/10
   Epoch 028/100 | Train: 2.1568 | Val: 2.1577 | LR: 9.15e-04 | Time: 68.8s
      ✓ Val loss improved (2.1580 → 2.1577). Savi

   Epoch 008/100 | Train: 2.0581 | Val: 2.0421 | LR: 7.30e-04 | Time: 98.5s
      ✓ Val loss improved (2.0528 → 2.0421). Saving...
   Epoch 009/100 | Train: 2.0460 | Val: 2.0374 | LR: 8.20e-04 | Time: 97.6s
      ✓ Val loss improved (2.0421 → 2.0374). Saving...
   Epoch 010/100 | Train: 2.0390 | Val: 2.0294 | LR: 9.10e-04 | Time: 97.8s
      ✓ Val loss improved (2.0374 → 2.0294). Saving...
   Epoch 011/100 | Train: 2.0333 | Val: 2.0249 | LR: 1.00e-03 | Time: 98.7s
      ✓ Val loss improved (2.0294 → 2.0249). Saving...
   Epoch 012/100 | Train: 2.0285 | Val: 2.0215 | LR: 1.00e-03 | Time: 98.1s
      ✓ Val loss improved (2.0249 → 2.0215). Saving...
   Epoch 013/100 | Train: 2.0249 | Val: 2.0184 | LR: 9.99e-04 | Time: 98.4s
      ✓ Val loss improved (2.0215 → 2.0184). Saving...
   Epoch 014/100 | Train: 2.0213 | Val: 2.0162 | LR: 9.97e-04 | Time: 97.7s
      ✓ Val loss improved (2.0184 → 2.0162). Saving...
   Epoch 015/100 | Train: 2.0182 | Val: 2.0137 | LR: 9.95e-04 | Time: 97.7s
      ✓

...

   Epoch 045/100 | Train: 1.1308 | Val: 1.1224 | LR: 6.88e-04 | Time: 443.1s
      ✓ Val loss improved (1.1232 → 1.1224). Saving...
   Epoch 046/100 | Train: 1.1299 | Val: 1.1229 | LR: 6.71e-04 | Time: 440.3s
      EarlyStopping counter: 1/10
   Epoch 047/100 | Train: 1.1292 | Val: 1.1233 | LR: 6.55e-04 | Time: 444.2s
      EarlyStopping counter: 2/10
   Epoch 048/100 | Train: 1.1281 | Val: 1.1209 | LR: 6.38e-04 | Time: 441.4s
      ✓ Val loss improved (1.1224 → 1.1209). Saving...
   Epoch 049/100 | Train: 1.1268 | Val: 1.1216 | LR: 6.21e-04 | Time: 440.2s
      EarlyStopping counter: 1/10
   Epoch 050/100 | Train: 1.1263 | Val: 1.1217 | LR: 6.04e-04 | Time: 440.6s
      EarlyStopping counter: 2/10
   Epoch 051/100 | Train: 1.1254 | Val: 1.1208 | LR: 5.87e-04 | Time: 442.0s
      ✓ Val loss improved (1.1209 → 1.1208). Saving...
   Epoch 052/100 | Train: 1.1244 | Val: 1.1180 | LR: 5.70e-04 | Time: 441.3s
      ✓ Val loss improved (1.1208 → 1.1180). Saving...
   Epoch 053/100 | Train: 1.

   Epoch 004/100 | Train: 1.2746 | Val: 1.2032 | LR: 3.70e-04 | Time: 584.4s
      ✓ Val loss improved (1.3052 → 1.2032). Saving...
   Epoch 005/100 | Train: 1.2056 | Val: 1.1636 | LR: 4.60e-04 | Time: 584.3s
      ✓ Val loss improved (1.2032 → 1.1636). Saving...
   Epoch 006/100 | Train: 1.1679 | Val: 1.1258 | LR: 5.50e-04 | Time: 585.4s
      ✓ Val loss improved (1.1636 → 1.1258). Saving...
   Epoch 007/100 | Train: 1.1437 | Val: 1.1057 | LR: 6.40e-04 | Time: 584.7s
      ✓ Val loss improved (1.1258 → 1.1057). Saving...
   Epoch 008/100 | Train: 1.1221 | Val: 1.1019 | LR: 7.30e-04 | Time: 584.1s
      ✓ Val loss improved (1.1057 → 1.1019). Saving...
   Epoch 009/100 | Train: 1.1014 | Val: 1.0735 | LR: 8.20e-04 | Time: 583.1s
      ✓ Val loss improved (1.1019 → 1.0735). Saving...
   Epoch 010/100 | Train: 1.0885 | Val: 1.0600 | LR: 9.10e-04 | Time: 581.4s
      ✓ Val loss improved (1.0735 → 1.0600). Saving...
   Epoch 011/100 | Train: 1.0742 | Val: 1.0484 | LR: 1.00e-03 | Time: 582.7s

   Epoch 068/100 | Train: 0.9474 | Val: 0.9437 | LR: 2.97e-04 | Time: 582.7s
      EarlyStopping counter: 1/10
   Epoch 069/100 | Train: 0.9467 | Val: 0.9417 | LR: 2.82e-04 | Time: 579.1s
      ✓ Val loss improved (0.9419 → 0.9417). Saving...
   Epoch 070/100 | Train: 0.9465 | Val: 0.9416 | LR: 2.66e-04 | Time: 583.4s
      ✓ Val loss improved (0.9417 → 0.9416). Saving...
   Epoch 071/100 | Train: 0.9456 | Val: 0.9409 | LR: 2.51e-04 | Time: 584.0s
      ✓ Val loss improved (0.9416 → 0.9409). Saving...
   Epoch 072/100 | Train: 0.9455 | Val: 0.9408 | LR: 2.36e-04 | Time: 584.3s
      ✓ Val loss improved (0.9409 → 0.9408). Saving...
   Epoch 073/100 | Train: 0.9448 | Val: 0.9414 | LR: 2.21e-04 | Time: 589.1s
      EarlyStopping counter: 1/10
   Epoch 074/100 | Train: 0.9448 | Val: 0.9407 | LR: 2.07e-04 | Time: 584.8s
      ✓ Val loss improved (0.9408 → 0.9407). Saving...
   Epoch 075/100 | Train: 0.9443 | Val: 0.9404 | LR: 1.93e-04 | Time: 584.3s
      ✓ Val loss improved (0.9407 → 0.940

   Epoch 026/100 | Train: 0.8746 | Val: 0.8599 | LR: 9.33e-04 | Time: 723.1s
      ✓ Val loss improved (0.8642 → 0.8599). Saving...
   Epoch 027/100 | Train: 0.8711 | Val: 0.8583 | LR: 9.24e-04 | Time: 722.7s
      ✓ Val loss improved (0.8599 → 0.8583). Saving...
   Epoch 028/100 | Train: 0.8674 | Val: 0.8536 | LR: 9.15e-04 | Time: 723.2s
      ✓ Val loss improved (0.8583 → 0.8536). Saving...
   Epoch 029/100 | Train: 0.8642 | Val: 0.8511 | LR: 9.05e-04 | Time: 724.2s
      ✓ Val loss improved (0.8536 → 0.8511). Saving...
   Epoch 030/100 | Train: 0.8610 | Val: 0.8467 | LR: 8.94e-04 | Time: 724.5s
      ✓ Val loss improved (0.8511 → 0.8467). Saving...
   Epoch 031/100 | Train: 0.8599 | Val: 0.8451 | LR: 8.83e-04 | Time: 726.0s
      ✓ Val loss improved (0.8467 → 0.8451). Saving...
   Epoch 032/100 | Train: 0.8564 | Val: 0.8446 | LR: 8.72e-04 | Time: 726.2s
      ✓ Val loss improved (0.8451 → 0.8446). Saving...
   Epoch 033/100 | Train: 0.8545 | Val: 0.8404 | LR: 8.60e-04 | Time: 724.7s

   Epoch 091/100 | Train: 0.8122 | Val: 0.8069 | LR: 3.11e-05 | Time: 716.2s
      EarlyStopping counter: 1/10
   Epoch 092/100 | Train: 0.8121 | Val: 0.8065 | LR: 2.54e-05 | Time: 715.6s
      ✓ Val loss improved (0.8066 → 0.8065). Saving...
   Epoch 093/100 | Train: 0.8118 | Val: 0.8066 | LR: 2.03e-05 | Time: 718.1s
      EarlyStopping counter: 1/10
   Epoch 094/100 | Train: 0.8118 | Val: 0.8064 | LR: 1.58e-05 | Time: 723.8s
      ✓ Val loss improved (0.8065 → 0.8064). Saving...
   Epoch 095/100 | Train: 0.8116 | Val: 0.8068 | LR: 1.19e-05 | Time: 729.2s
      EarlyStopping counter: 1/10
   Epoch 096/100 | Train: 0.8117 | Val: 0.8064 | LR: 8.59e-06 | Time: 722.8s
      ✓ Val loss improved (0.8064 → 0.8064). Saving...
   Epoch 097/100 | Train: 0.8116 | Val: 0.8064 | LR: 5.86e-06 | Time: 730.8s
      EarlyStopping counter: 1/10
   Epoch 098/100 | Train: 0.8116 | Val: 0.8063 | LR: 3.74e-06 | Time: 733.7s
      ✓ Val loss improved (0.8064 → 0.8063). Saving...
   Epoch 099/100 | Train: 0.

In [17]:
# In[18]:

# =============================================================================
# CELL 18: SAVE FINAL RESULTS & PLOT
# =============================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS SUMMARY")
print("="*70)

# Save results to JSON
results_json_path = os.path.join(CONFIG['results_dir'], "experiment_results.json")
with open(results_json_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"💾 Results saved: {results_json_path}")

# Print table
print(f"\n   Error Model: {CONFIG['error_name']} ({CONFIG['platform']}), Seq Length: {CONFIG['seq_length']}")
print(f"   {'M':<8} {'Bi-LSTM':<12} {'Min.Dist':<12} {'KL Div':<12} {'Max.Like':<12}")
print(f"   {'-'*56}")
for i, M in enumerate(results['coverage']):
    print(f"   {M:<8} {results['lstm'][i]:<12.2f} {results['mindist'][i]:<12.2f} "
          f"{results['kl'][i]:<12.2f} {results['ml'][i]:<12.2f}")
print(f"   {'='*56}")

# Final comparison plot
plot_path = os.path.join(CONFIG['results_dir'], "final_comparison_plot.png")
plot_comparison_results(results, plot_path, CONFIG)

print(f"\n✅ All experiments completed!")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['platform']})")
print(f"   Results directory: {CONFIG['results_dir']}")




📊 FINAL RESULTS SUMMARY
💾 Results saved: ./results/NP22_2mix_3mix_4mix/experiment_results.json

   Error Model: NP22 (Nanopore Pilot Nov-2022), Seq Length: 136
   M        Bi-LSTM      Min.Dist     KL Div       Max.Like    
   --------------------------------------------------------
   1        18.58        18.57        18.57        18.57       
   2        27.61        26.10        26.10        26.10       
   3        33.36        29.90        29.90        29.90       
   5        40.13        34.39        31.49        31.49       
   8        47.57        40.92        33.22        33.22       
   10       51.62        43.24        38.03        38.03       
   15       59.74        47.80        40.72        40.72       
   20       65.68        50.14        39.04        39.04       
   25       70.29        52.55        39.49        39.49       
📈 Comparison plot saved: ./results/NP22_2mix_3mix_4mix/final_comparison_plot.png

✅ All experiments completed!
   Error Model: NP22 (Nanopo